# Ninai × LangChain Adapter

Wires the **Ninai Python SDK** into a LangChain pipeline:

1. `NinaiChatMessageHistory` — persists conversation turns in Ninai memory
2. `NinaiSearchTool` — semantic search as a LangChain `BaseTool`
3. `NinaiMemoryTool` — store facts from within a LangChain agent

All API calls are **mocked** — runs offline without a live Ninai server.

In [1]:
import sys, os
SDK_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'sdk', 'python'))
if SDK_PATH not in sys.path:
    sys.path.insert(0, SDK_PATH)
import langchain_core, ninai
print('langchain_core:', langchain_core.__version__)
print('ninai SDK:', ninai.__version__)


langchain_core: 1.2.28
ninai SDK: 1.0.0


## Mock Ninai client

In [2]:
from unittest.mock import MagicMock
from types import SimpleNamespace

_STORE: dict = {}
_CTR = [0]

def _mem(id, content, title='', tags=None):
    return SimpleNamespace(id=id, content=content, title=title, tags=tags or [])

def _mock_create(**kwargs):
    _CTR[0] += 1
    m = _mem(str(_CTR[0]), kwargs.get('content', ''), kwargs.get('title', ''), kwargs.get('tags', []))
    _STORE[m.id] = m
    return m

def _mock_search(query, **kwargs):
    items = [SimpleNamespace(memory_id=m.id, content=m.content, score=0.9, title=m.title)
             for m in list(_STORE.values())[-5:]]
    return SimpleNamespace(items=items[:3], total=len(items))

# Use a plain MagicMock so .memories auto-creates child mocks
client = MagicMock()
client.memories.create.side_effect = _mock_create
client.memories.search.side_effect = _mock_search
print('Mock client ready')


Mock client ready


## 1. NinaiChatMessageHistory

In [3]:
from typing import List, Sequence
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage


class NinaiChatMessageHistory(BaseChatMessageHistory):
    """Persists LangChain conversation turns in Ninai memory."""

    def __init__(self, session_id: str, ninai_client):
        self.session_id = session_id
        self._client = ninai_client
        self._messages: List[BaseMessage] = []

    @property
    def messages(self) -> List[BaseMessage]:
        return list(self._messages)

    def add_messages(self, messages: Sequence[BaseMessage]) -> None:
        for msg in messages:
            role = 'user' if isinstance(msg, HumanMessage) else 'assistant'
            self._client.memories.create(
                content=msg.content,
                title=f'[{role}] session:{self.session_id}',
                tags=['conversation', f'session:{self.session_id}', role],
            )
            self._messages.append(msg)

    def clear(self) -> None:
        self._messages.clear()


history = NinaiChatMessageHistory(session_id='demo-001', ninai_client=client)
history.add_messages([HumanMessage(content='What is Ninai?')])
history.add_messages([AIMessage(content='Ninai is a Cognitive OS for enterprise.')])
print(f'Messages: {len(history.messages)}, Ninai writes: {client.memories.create.call_count}')


Messages: 2, Ninai writes: 2


## 2. LangChain Tools

In [4]:
from typing import Optional, Type
from langchain_core.tools import BaseTool
from langchain_core.callbacks import CallbackManagerForToolRun
from pydantic import BaseModel, Field


class _SearchInput(BaseModel):
    query: str = Field(description='Semantic search query')
    top_k: int = Field(default=5)


class NinaiSearchTool(BaseTool):
    name: str = 'ninai_search'
    description: str = 'Search organisational memory for relevant context.'
    args_schema: Type[BaseModel] = _SearchInput
    ninai_client: object = None

    class Config:
        arbitrary_types_allowed = True

    def _run(self, query: str, top_k: int = 5,
             run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        res = self.ninai_client.memories.search(query, limit=top_k)
        if not res.items:
            return 'No relevant memories found.'
        return '\n'.join(f'- [{r.score:.2f}] {r.content}' for r in res.items)


class _StoreInput(BaseModel):
    content: str = Field(description='Content to store')
    tags: str = Field(default='')


class NinaiMemoryTool(BaseTool):
    name: str = 'ninai_remember'
    description: str = 'Store an important fact in Ninai organisational memory.'
    args_schema: Type[BaseModel] = _StoreInput
    ninai_client: object = None

    class Config:
        arbitrary_types_allowed = True

    def _run(self, content: str, tags: str = '',
             run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        tag_list = [t.strip() for t in tags.split(',') if t.strip()]
        mem = self.ninai_client.memories.create(content=content, tags=tag_list)
        return f'Stored memory {mem.id}: {content[:80]}'


search_tool = NinaiSearchTool(ninai_client=client)
remember_tool = NinaiMemoryTool(ninai_client=client)

remember_tool._run('Q3 revenue target is $4.2M', tags='finance,Q3')
remember_tool._run('Deploy freeze starts 2026-04-15', tags='ops,deploy')

print('Search results:')
print(search_tool._run('revenue target'))


Search results:
- [0.90] What is Ninai?
- [0.90] Ninai is a Cognitive OS for enterprise.
- [0.90] Q3 revenue target is $4.2M


C:\Users\selva\AppData\Local\Temp\ipykernel_33052\1186122258.py:12: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class NinaiSearchTool(BaseTool):
C:\Users\selva\AppData\Local\Temp\ipykernel_33052\1186122258.py:34: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class NinaiMemoryTool(BaseTool):


## 3. RunnableWithMessageHistory chain

In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory


def stub_llm(prompt_value):
    msgs = prompt_value.to_messages()
    user_text = next((m.content for m in reversed(msgs) if isinstance(m, HumanMessage)), '')
    return AIMessage(content=f'[stub] Received: "{user_text}"')


chain_with_history = RunnableWithMessageHistory(
    ChatPromptTemplate.from_messages([
        ('system', 'You are a helpful enterprise assistant powered by Ninai.'),
        MessagesPlaceholder(variable_name='history'),
        ('human', '{input}'),
    ]) | RunnableLambda(stub_llm),
    lambda sid: NinaiChatMessageHistory(session_id=sid, ninai_client=client),
    input_messages_key='input',
    history_messages_key='history',
)

cfg = {'configurable': {'session_id': 'lc-42'}}
r1 = chain_with_history.invoke({'input': 'What is our Q3 revenue target?'}, config=cfg)
r2 = chain_with_history.invoke({'input': 'When is the next deploy freeze?'}, config=cfg)
print('Turn 1:', r1.content)
print('Turn 2:', r2.content)
print('\nLangChain + Ninai adapter verified.')


Turn 1: [stub] Received: "What is our Q3 revenue target?"
Turn 2: [stub] Received: "When is the next deploy freeze?"

LangChain + Ninai adapter verified.


## 4. Toolkit factory

In [6]:
def get_ninai_toolkit(ninai_client):
    """Return all Ninai tools ready for a LangChain agent executor."""
    return [NinaiSearchTool(ninai_client=ninai_client), NinaiMemoryTool(ninai_client=ninai_client)]

for t in get_ninai_toolkit(client):
    print(f'{t.name:22s} {t.description}')


ninai_search           Search organisational memory for relevant context.
ninai_remember         Store an important fact in Ninai organisational memory.
